# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset package using the `mlcroissant` library, following FAIR data practices.

### Dataset Source
This dataset is described by a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata as an object
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Display the available record sets and their fields using their `@id`s.

In [ ]:
from mlcroissant.types import RecordSet

# List available record sets by @id
print("Record sets available:")
for rs in dataset.record_sets:
    print(f"  - {rs['@id']}")

# For each record set, display its fields and columns by @id
for rs in dataset.record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    # Display field @id's
    if 'field' in rs:
        print("  Fields:")
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            field_obj = dataset._schema_node(field)
            print(f"    - {field_obj['@id']} ({field_obj.get('name', '')})")
            # If field has columns
            if 'column' in field_obj:
                columns = field_obj['column'] if isinstance(field_obj['column'], list) else [field_obj['column']]
                for col in columns:
                    col_obj = dataset._schema_node(col)
                    print(f"        - Column: {col_obj['@id']} ({col_obj.get('name', '')})")

## 3. Data Extraction
Load records from each record set into a DataFrame for analysis. Each record set and its fields/columns are referenced by `@id` as required.

In [ ]:
# First, get the list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# For demonstration, extract all records into a dictionary of DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"{len(df)} records loaded. Columns with IDs:")
        print(df.columns.tolist())
    else:
        print("No records found in this record set.")
# For convenience, pick the first non-empty record set for following analysis
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break
if main_rs_id is not None:
    print(f"\nUsing record set '{main_rs_id}' for EDA below: ")
    display(dataframes[main_rs_id].head())
else:
    print("No record sets with data were found.")

## 4. Exploratory Data Analysis (EDA)
Let's clean and analyze the main record set by filtering, normalizing, and grouping data fields by their `@id`s.
> **Note:** For demonstration, we select the first numeric field in the first available DataFrame. You may replace these with specific `@id`s relevant to your analysis.

In [ ]:
import numpy as np

# Identify a numeric field by brute-force (replace with domain knowledge if available):
df = dataframes.get(main_rs_id)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError('No numeric field found in current DataFrame.')
group_field_id = None
# Attempt to find a suitable grouping field (string/categorical field)
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == object:
        nunique = df[col].nunique(dropna=True)
        if 1 < nunique < 20:
            group_field_id = col
            break

threshold = df[numeric_field_id].dropna().quantile(0.75)  # Use 75th percentile as demo threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
display(filtered_df[[numeric_field_id] + ([group_field_id] if group_field_id else [])].head())

# Normalize the numeric field in the filtered dataframe
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by the group_field_id if available
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
    print(f"\nGrouped data by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its grouping. You can adapt plot code and axes labels as needed for specific field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

if group_field_id is not None:
    plt.figure(figsize=(10, 5))
    ax = sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and described the FAIR^2 dataset using its Croissant schema.
- Explored available record sets and fields by their `@id` for reproducible and unambiguous reference.
- Loaded tabular data, applied basic filtering, normalization, and grouping for exploratory analysis.
- Visualized distributions and group effects.

_For further analysis, refine field selections using the `@id`s from section 2 to match your domain questions, and expand EDA and modeling as required._